# Ngân sách RAM 4GB — SLM + STT + TTS + CV
Chọn mỗi tác vụ một model từ kết quả đã đo, rồi cộng RAM.
- **Đồng thời**: mọi thành phần nằm trong RAM cùng lúc.
- **Tuần tự**: STT → SLM → TTS không cần cùng lúc; CV chỉ nạp khi cần. Đỉnh = OS + max(các bước).

In [1]:
import sys; sys.path.insert(0, ".")
from bench_common import *

# ---- ĐIỀN SỐ CỦA BẠN ----
TOTAL_MB       = 4096
OS_OVERHEAD_MB = 700     # OS + service nền trên UNO Q (đo lại trên thiết bị)
SLM_MB         = 600     # lấy từ edge_slm_benchmark (RAM của model đã chọn)
PICK = {                 # tên model đúng như cột `model` trong bảng kết quả
    "stt": ("stt", "faster-whisper tiny.en int8"),
    "tts": ("tts", "piper amy-low"),
    "cv_det": ("cv_detection_ram", "YOLO11n-320"),
    "cv_ocr": ("cv_ocr", "RapidOCR (onnx)"),
}

def ram_of(subdir, name):
    df = table(subdir)
    r = df[df["model"] == name]
    if r.empty or "ram_total_peak_mb" not in r:
        raise SystemExit(f"Chưa có kết quả cho {subdir}/{name}")
    return float(r.iloc[0]["ram_total_peak_mb"])

parts = {k: ram_of(*v) for k, v in PICK.items()}
parts["slm"] = SLM_MB
for k, v in parts.items():
    print(f"{k:8s} {v:8.0f} MB")

concurrent = OS_OVERHEAD_MB + sum(parts.values())
sequential = OS_OVERHEAD_MB + max(parts["stt"], parts["tts"], parts["slm"]) + max(parts["cv_det"], parts["cv_ocr"])
print(f"\nĐồng thời : {concurrent:.0f} MB  ({concurrent / TOTAL_MB:.0%} của {TOTAL_MB} MB)")
print(f"Tuần tự   : {sequential:.0f} MB  ({sequential / TOTAL_MB:.0%})")
print("→ vừa" if concurrent < 0.8 * TOTAL_MB else "→ NÊN load/unload theo nhu cầu (chạy tuần tự)")

stt           246 MB
tts           234 MB
cv_det         76 MB
cv_ocr        501 MB
slm           600 MB

Đồng thời : 2358 MB  (58% của 4096 MB)
Tuần tự   : 1801 MB  (44%)
→ vừa


**Cảnh báo:** `ram_total_peak_mb` đo trên PC x86 và gồm runtime của chính nó (ort/ctranslate2...). Khi chạy chung một tiến trình các runtime có thể dùng chung thư viện (nhỏ hơn tổng), nhưng ARM có thể khác. Hãy lặp lại phép đo trên UNO Q và chừa dư ≥ 20% RAM.